# Compare CHUQ and KDUQ potentials

In [1]:
import pickle

import numpy as np

In [2]:
import matplotlib

matplotlib.use("pgf")
from matplotlib import pyplot as plt
from matplotlib import rcParams
from matplotlib.lines import Line2D

colors = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
    "#bcbd22",
    "#17becf",
]


rcParams["legend.fontsize"] = 14
rcParams["font.size"] = 14
rcParams["font.weight"] = "normal"
rcParams["xtick.labelsize"] = 14.0
rcParams["ytick.labelsize"] = 14.0
rcParams["lines.linewidth"] = 2.0
rcParams["xtick.major.pad"] = "10"
rcParams["ytick.major.pad"] = "10"

In [3]:
from jitr.reactions import ElasticReaction

In [4]:
from jitr.optical_potentials import chuq, kduq, wlh

In [5]:
neutron = (1, 0)
proton = (1, 1)

In [6]:
target = (54, 26)
projectile = proton
energy_lab = 65
rxn = ElasticReaction(target=target, projectile=projectile)

In [7]:
# kduq_samples = kduq.get_samples_federal(projectile)
# chuq_samples = chuq.get_samples_federal(projectile)

In [8]:
kduq_samples = kduq.get_samples_democratic(projectile)
chuq_samples = chuq.get_samples_democratic()
wlh_samples = wlh.get_samples(projectile)

In [9]:
kinematics = rxn.kinematics(energy_lab)
kinematics

ChannelKinematics(Elab=65, Ecm=63.80812277524172, mu=np.float64(981.8605612726117), k=np.float64(1.7650843754385006), eta=np.float64(0.5348556066621344))

In [10]:
r = np.linspace(0.1, 10, 100)
chuq_v_central = np.zeros((chuq.NUM_POSTERIOR_SAMPLES, 100), dtype=complex)
kduq_v_central = np.zeros((kduq.NUM_POSTERIOR_SAMPLES, 100), dtype=complex)
wlh_v_central = np.zeros((wlh.NUM_POSTERIOR_SAMPLES, 100), dtype=complex)

chuq_v_so = np.zeros((chuq.NUM_POSTERIOR_SAMPLES, 100), dtype=complex)
kduq_v_so = np.zeros((kduq.NUM_POSTERIOR_SAMPLES, 100), dtype=complex)
wlh_v_so = np.zeros((wlh.NUM_POSTERIOR_SAMPLES, 100), dtype=complex)

for i, kduq_sample in enumerate(kduq_samples):
    coul, cent, so = kduq.calculate_params(
        projectile, target, kinematics.Elab, *kduq_sample
    )
    kduq_v_central[i, :] = kduq.central(r, *cent)
    kduq_v_so[i, :] = kduq.spin_orbit(r, *so)

for i, chuq_sample in enumerate(chuq_samples):
    coul, cent, so = chuq.calculate_params(
        projectile, target, kinematics.Elab, *chuq_sample
    )
    chuq_v_central[i, :] = chuq.central(r, *cent)
    chuq_v_so[i, :] = chuq.spin_orbit(r, *so)

for i, wlh_sample in enumerate(wlh_samples):
    coul, cent, so = wlh.calculate_params(
        projectile, target, kinematics.Elab, *wlh_sample
    )
    wlh_v_central[i, :] = wlh.central(r, *cent)
    wlh_v_so[i, :] = wlh.spin_orbit(r, *so)

/mnt/ffs24/home/beyerkyl/jitr/src/jitr/optical_potentials/kduq.py:397: RuntimeWarning: overflow encountered in exp
  d2 = d2_0 + d2_A / (1 + np.exp((A - d2_A3) / d2_A2))


In [11]:
def get_ci(x):
    return np.percentile(x.real, [5, 50, 95], axis=0) + 1j * np.percentile(
        x.imag, [5, 50, 95], axis=0
    )

In [12]:
chuq_v_central = get_ci(chuq_v_central)
chuq_v_so = get_ci(chuq_v_so)
kduq_v_central = get_ci(kduq_v_central)
kduq_v_so = get_ci(kduq_v_so)
wlh_v_central = get_ci(wlh_v_central)
wlh_v_so = get_ci(wlh_v_so)

In [17]:
fig = plt.figure()
# plt.plot(r, chuq_v_so[1].real, color="tab:blue")
plt.fill_between(r, chuq_v_so[0].real, chuq_v_so[2].real, alpha=0.3, color="tab:blue")

plt.fill_between(
    r, chuq_v_so[0].imag, chuq_v_so[2].imag, hatch="|-|-|-", alpha=0.3, color="tab:blue"
)

plt.fill_between(r, kduq_v_so[0].real, kduq_v_so[2].real, alpha=0.3, color="#ff4500")

# plt.plot(r, kduq_v_so[1].imag, "--", color="#ff4500")
plt.fill_between(
    r, kduq_v_so[0].imag, kduq_v_so[2].imag, hatch="|-|-|-", alpha=0.3, color="#ff4500"
)


# plt.plot(r, wlh_v_so[1].real,  ":", color="m")
plt.fill_between(r, wlh_v_so[0].real, wlh_v_so[2].real, alpha=0.2, color="m")


plt.fill_between(
    r, wlh_v_so[0].imag, wlh_v_so[2].imag, hatch="|-|-|-", alpha=0.2, color="m"
)


plt.ylabel(r"$U_{so}(r)$ [MeV]")
plt.xlabel(r"$r$ [fm]")
plt.tight_layout()
plt.savefig("./iron54_so.pgf", format="pgf")

In [18]:
fig = plt.figure()

# (p,) = plt.plot(r, chuq_v_central[1].real, color="tab:blue", ":")
plt.fill_between(
    r,
    chuq_v_central[0].real,
    chuq_v_central[2].real,
    alpha=0.3,
    color="tab:blue",
    zorder=99,
    label="CHUQ",
)

# plt.plot(r, chuq_v_central[1].imag,have "--", color="tab:blue")
plt.fill_between(
    r,
    chuq_v_central[0].imag,
    chuq_v_central[2].imag,
    hatch="|-|-|-",
    alpha=0.3,
    color="tab:blue",
    zorder=99,
)

# (p,) = plt.plot(r, kduq_v_central[1].real, ":")
plt.fill_between(
    r,
    kduq_v_central[0].real,
    kduq_v_central[2].real,
    alpha=0.3,
    color="#ff4500",
    label="KDUQ",
)

# plt.plot(r, kduq_v_central[1].imag, "--", color=p.get_color())
plt.fill_between(
    r,
    kduq_v_central[0].imag,
    kduq_v_central[2].imag,
    hatch="|-|-|-",
    alpha=0.3,
    color="#ff4500",
)

# (p,) = plt.plot(r, wlh_v_central[1].real, ":")
plt.fill_between(
    r, wlh_v_central[0].real, wlh_v_central[2].real, alpha=0.2, color="m", label="WLH"
)

# plt.plot(r, wlh_v_central[1].imag, "--", color=p.get_color())
plt.fill_between(
    r,
    wlh_v_central[0].imag,
    wlh_v_central[2].imag,
    hatch="|-|-|-",
    alpha=0.2,
    color="m",
)

plt.legend()

plt.ylabel(r"$U(r)$ [MeV]")
plt.xlabel(r"$r$ [fm]")
plt.tight_layout()

plt.savefig("./iron54_central.pgf", format="pgf")